# 02. Building the job search index (FAISS)

**Goal:** show how the job index is built and searched. The real index was built once with `build_index.py`.
This notebook builds a small copy in memory, so the real index is never touched.

In [1]:
import os, sys
os.environ["TQDM_DISABLE"] = "1"     # hide progress bars so the notebook stays tidy
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print("Working folder:", ROOT)

Working folder: C:\Users\pramod\Smarthire-genai


## Step 1: look at the job data

In [2]:
import pandas as pd
from src.config import JOBS_PATH
from src.search.job_search import make_job_text

jobs = pd.read_csv(JOBS_PATH).dropna(subset=["Job_Title", "Skills_Required"]).reset_index(drop=True)
print("Jobs:", len(jobs))
print("Columns:", list(jobs.columns))
jobs[["Job_Title", "Company", "Experience_Level", "Skills_Required"]].head(5)

C:\Users\pramod\Smarthire-genai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Jobs: 5000
Columns: ['Job_ID', 'Job_Title', 'Company', 'Company_Type', 'Industry', 'City', 'Location_Tier', 'Experience_Level', 'Job_Type', 'Work_Mode', 'Salary_LPA', 'Skills_Required', 'Education_Required', 'Openings', 'Applicants', 'Company_Rating', 'Date_Posted']


,Job_Title,Company,Experience_Level,Skills_Required
0,Android Developer,Tech Mahindra,Senior (6-10 yrs),"Kotlin, Java, REST APIs"
1,QA Engineer,Dream11,Senior (6-10 yrs),"Selenium, Manual Testing, Postman, API Testing..."
2,Business Analyst,HAL,Senior (6-10 yrs),"JIRA, Excel, Power BI"
3,Cybersecurity Analyst,Groww,Mid (3-6 yrs),"Penetration Testing, Python, Ethical Hacking, ..."
4,Python Developer,Oracle,Junior (1-3 yrs),"Docker, REST APIs, AWS, PostgreSQL"


## Step 2: the text that describes each job
Each job becomes one short text: title, skills, industry, level and education.

In [3]:
print(make_job_text(jobs.iloc[0]))

Android Developer. Skills: Kotlin, Java, REST APIs. Industry: Information Technology. Experience: Senior (6-10 yrs). Education: M.Tech/M.E..


## Step 3: build a small demo index (300 jobs, in memory)

In [4]:
import faiss
from src.search.embed import embed_texts

sample = jobs.head(300)
texts = [make_job_text(r) for _, r in sample.iterrows()]
vectors = embed_texts(texts)
index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)
print("Jobs stored in the demo index:", index.ntotal)
print("Numbers per job:", vectors.shape[1])

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:  20%|██        | 1/5 [00:00<00:01,  2.44it/s]

Batches:  40%|████      | 2/5 [00:00<00:01,  2.58it/s]

Batches:  60%|██████    | 3/5 [00:01<00:00,  2.72it/s]

Batches:  80%|████████  | 4/5 [00:01<00:00,  2.89it/s]

Batches: 100%|██████████| 5/5 [00:01<00:00,  3.40it/s]

Batches: 100%|██████████| 5/5 [00:01<00:00,  3.05it/s]

Jobs stored in the demo index: 300
Numbers per job: 384


## Step 4: search the demo index

In [5]:
query = embed_texts(["Python SQL Excel Power BI data analyst fresher"])
scores, ids = index.search(query, 5)
for s, i in zip(scores[0], ids[0]):
    row = sample.iloc[int(i)]
    print(round(float(s), 3), "|", row["Job_Title"], "|", row["Skills_Required"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 166.65it/s]

0.567 | Power BI Developer | Power BI, SQL, Excel, DAX, Data Modeling
0.56 | Power BI Developer | Data Modeling, SQL, ETL, Excel, Power BI, DAX
0.547 | Power BI Developer | Azure, SQL, Data Modeling, DAX
0.537 | Data Analyst | Power BI, SQL, Data Visualization, Tableau, Excel
0.536 | Data Analyst | Python, Tableau, Power BI


## Step 5: the real index
The real index holds all the jobs. It is saved on disk and loaded in a moment, so jobs are never turned into numbers twice.

In [6]:
import os
from src.config import INDEX_PATH
from src.search.job_search import search_jobs

real = faiss.read_index(INDEX_PATH)
print("Real index:", real.ntotal, "jobs,", round(os.path.getsize(INDEX_PATH) / 1_000_000, 1), "MB on disk")
for r in search_jobs("Python SQL Excel Power BI data analyst fresher", top_n=5):
    print(r["score"], "|", r["Job_Title"], "|", r["Company"], "|", r["Experience_Level"])

Real index: 5000 jobs, 7.7 MB on disk


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 169.81it/s]

0.583 | Data Analyst | Leadsquared | Fresher (0-1 yr)
0.578 | Power BI Developer | Microsoft | Mid (3-6 yrs)
0.577 | Power BI Developer | Byju's | Junior (1-3 yrs)
0.576 | Power BI Developer | Ola | Mid (3-6 yrs)
0.574 | Power BI Developer | Leadsquared | Senior (6-10 yrs)


## What I learned
- Building the index is slow but done once. Searching it takes a few milliseconds.
- The search result depends on which job details go into the text, so I include title, skills, level and education.